# D3 · Sistema multi-agente con LangGraph (Ministral 3 3B)

**Entregable 3 · Generative Artificial Intelligence.** Corre en Google Colab con `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)` y después `Ejecutar todo`.

Cuatro agentes con el mismo modelo (`ministral-3:3b-instruct-2512-q4_K_M`, igual que D2): **Rastreador de temas → Decisiones + Tareas → Auditor**, orquestados con LangGraph. Cada agente decide en cada paso si llama una herramienta o entrega (acción en JSON restringido), y el Auditor puede devolver ítems a su autor.

El notebook clona el repositorio, instala Ollama y corre las condiciones con `python -m agentes.correr`. Los resultados se guardan tras cada reunión: si la sesión se corta, vuelve a ejecutar y se reanuda.

**Para repartir el tiempo entre varias sesiones de Colab**, cambia `CONDICIONES` en la celda de parámetros (por ejemplo una sesión con `full sin_fewshot` y otra con `sin_tools sin_auditor sin_rastreador`) y junta las carpetas `resultados/` al final. Todas deben correr en T4.

In [ ]:
# Parámetros
REPO = 'https://github.com/benjaminnalvear-dev/genai-vio-meeting-minutes.git'
RAMA = 'd3-multiagente'
# 'todas' = baseline structured d2_combined full sin_fewshot sin_tools sin_auditor sin_rastreador
CONDICIONES = 'todas'
REUNIONES = ''          # vacío = las 10 de test; por ejemplo 'M01 M02'
DEMO_VIDEO = False      # True: solo corre M01 con el sistema completo mostrando la conversación entre agentes

In [ ]:
# GPU, Ollama y modelo (igual que D2)
import subprocess, shutil, time, requests
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Esta sesión no tiene GPU. Entorno de ejecución > Cambiar tipo > GPU T4.')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
if not shutil.which('zstd'):
    subprocess.run(['apt-get','update','-qq'],check=True); subprocess.run(['apt-get','install','-y','-qq','zstd'],check=True)
if not shutil.which('ollama'):
    subprocess.run(['bash','-o','pipefail','-c','curl -fsSL https://ollama.com/install.sh | sh'],check=True)
server = subprocess.Popen(['ollama','serve'],stdout=open('/content/ollama.log','a'),stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags',timeout=2).ok: break
    except requests.RequestException: time.sleep(1)
else:
    raise RuntimeError('Ollama no inició; revisa /content/ollama.log')
subprocess.run(['ollama','pull','ministral-3:3b-instruct-2512-q4_K_M'],check=True)

In [ ]:
# Código del repositorio
import os
if not os.path.exists('/content/repo'):
    subprocess.run(['git','clone','--depth','1','--branch',RAMA,REPO,'/content/repo'],check=True)
os.chdir('/content/repo')
%pip -q install -r agentes/requirements.txt
!git log --oneline -1
!python -m unittest agentes.tests.test_agentes

In [ ]:
# Demo para el video: conversación entre agentes en pantalla
if DEMO_VIDEO:
    !python -m agentes.correr --reunion M01 --condicion full --en-vivo --salida /content/demo

In [ ]:
# Evaluación: condiciones × reuniones de test (se reanuda si se corta)
if not DEMO_VIDEO:
    args = ' '.join(f'--condicion {c}' for c in CONDICIONES.split())
    args += ' ' + ' '.join(f'--reunion {m}' for m in REUNIONES.split())
    !python -u -m agentes.correr {args}

In [ ]:
# Tabla con todo lo disponible y descarga de resultados + trazas
if not DEMO_VIDEO:
    !python -m agentes.correr --tabla
import shutil
from google.colab import files
carpeta = '/content/demo' if DEMO_VIDEO else 'resultados/d3/test'
archivo = shutil.make_archive('/content/d3_resultados', 'zip', root_dir=carpeta)
files.download(archivo)